# 01 - Business Understanding
**Lab AUEC Clustering | Defensa Papers ML UCB 2026**
Grupo JaimelA - Paper: *Autoencoded UMAP-Enhanced Clustering for Unsupervised Learning*
Chavooshi & Mamonov, 2025 - arXiv:2501.07729

---
## Objetivos del lab (CRISP-DM: Business Understanding)

| Objetivo | Descripcion |
|----------|-------------|
| **Replicar** el pipeline AUEC | Implementar CAE + UMAP + K-means sobre MNIST |
| **Comparar** con baselines | B1 K-means raw, B2 PCA+KMeans, B3 UMAP(raw)+KMeans |
| **Evaluar** cuantitativamente | ACC, NMI, ARI, Silhouette, Davies-Bouldin |
| **Identificar** contribucion de cada componente | Cuanto aporta el AE? Cuanto aporta UMAP? |

## El paper AUEC - Resumen

### Problema
El clustering no supervisado en espacios de alta dimension sufre de:
- *Curse of dimensionality*: las distancias euclidianas pierden significado
- Estructuras no lineales que KMeans no puede capturar

### Solucion propuesta
Pipeline de tres etapas:
```
Input (28x28) --> Autoencoder Convolucional --> Z (10D latente)
                                             --> UMAP (10D -> 10D)
                                             --> K-means / DBSCAN
```

### Loss del paper
El paper propone una loss compuesta:
`J = lambda * psi(spectral_gap) + rho * MSE`

**Simplificacion de este lab:** se usa solo MSE (la loss espectral requiere calcular
eigenvalores del Laplaciano del grafo por batch, inviable en CPU sin GPU dedicada).

## Arquitectura del Autoencoder Convolucional (CAE)

```
ENCODER
Input (28x28x1)
  -> Conv2D(32, 3x3, relu, same) + BatchNorm + MaxPool(2) -> 14x14x32
  -> Conv2D(64, 3x3, relu, same) + BatchNorm + MaxPool(2) -> 7x7x64
  -> Flatten (3136) -> Dense(256, relu) -> Dense(10) [espacio latente Z]

DECODER
  Dense(256, relu) -> Dense(7*7*64, relu) -> Reshape(7, 7, 64)
  -> Conv2DTranspose(64) + UpSampling(2) -> 14x14x64
  -> Conv2DTranspose(32) + UpSampling(2) -> 28x28x32
  -> Conv2DTranspose(1, sigmoid)          -> 28x28x1
```

## Hipotesis del experimento

**H1:** El pipeline AUEC (CAE + UMAP + KMeans) superara a los tres baselines en ACC y metricas internas.

**H2:** UMAP sobre el espacio latente del AE producira mejores clusters que UMAP directamente sobre pixeles.

**H3:** La estructura aprendida por el AE separara las 10 clases de MNIST mejor que PCA lineal.

## Relacion con la materia

| Tema de la materia | Aplicacion en este lab |
|--------------------|------------------------|
| Clustering no supervisado | K-means, DBSCAN sobre representaciones |
| Reduccion de dimensionalidad | PCA (baseline), UMAP (tecnica principal) |
| Deep Learning | Autoencoder Convolucional (encoder-decoder) |
| Evaluacion de modelos | Metricas internas y externas |
| CRISP-DM | Metodologia que estructura todo el lab |

In [ ]:
import sys, os
sys.path.insert(0, '..')
os.chdir('..')

from src.data_loader import load_config

cfg = load_config()
print("Configuracion cargada:")
for section, vals in cfg.items():
    print(f"  {section}: {vals}")

## Plan de ejecucion

1. `01_business_understanding.ipynb` <- **este notebook**
2. `02_data_understanding.ipynb` - Carga MNIST + EDA
3. `03_data_preparation.ipynb` - Preprocesamiento + persistencia
4. `04_modeling.ipynb` - Baselines + CAE + UMAP + Clustering
5. `05_evaluation.ipynb` - Metricas + visualizaciones + conclusiones

> Ejecutar en orden. Cada notebook carga los artefactos del anterior.